### Train Pipeline: Feature Selection And Optuna

Структура: all-features baseline -> holdout evaluation -> feature importance -> redundant analysis -> feature set experiments -> best feature set -> Optuna -> final model.

### 1. Imports / env / MLflow URI

In [ ]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.insert(0, '/home/gorelova_i_v/projects/cvm_churn-from-dac_binary-class_churn-dac/src')

from datetime import datetime

import os
from pathlib import Path
import joblib
from collections import defaultdict
from tqdm.auto import tqdm

import logging
logger = logging.getLogger()
logger.setLevel(logging.INFO)

import magpie.sql_utils as su

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)

import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.preprocessing import SplineTransformer
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance

import shap

from sklearn.metrics import (
    average_precision_score as sk_average_precision_score,
    roc_auc_score as sk_roc_auc_score,
    roc_curve,
    precision_recall_curve,
    PrecisionRecallDisplay,
    confusion_matrix as sk_confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    brier_score_loss as sk_brier_score_loss,
)

from cvm_ml_metrics.classification import (
    roc_auc_score,
    precision_recall_auc,
    confusion_matrix,
    log_loss,
    recall_score,
    precision_score,
    sensitivity_specificity,
    balanced_accuracy,
    f1_score,
    fbeta_score,
    brier_score_loss,
    youden_j,
    markedness,
    lift,
    gini,
    ks_stat_bin_class,
    ece_mce_fast,
    matthews_corrcoef,
    cohen_kappa_score,
)

import mlflow
from mlflow import MlflowClient
from mlflow.models.signature import infer_signature

In [ ]:
from cvm_model.io import State
import cvm_model.functions as func
import cvm_model.utils as utils
from cvm_model.parameters import (
    RANDOM_STATE,
    template,
    features,
    target,
    score,
    project_name,
    model_type,
    train_data_stat_suffix,
    artifacts_dir,
    jira,
    threshold,
)

# Path(artifacts_dir).mkdir(parents=True, exist_ok=True)

In [ ]:
# Если используешь .env, раскомментируй эти строки и перезапусти kernel/ячейку.
# %load_ext dotenv
# %dotenv
# %dotenv /Users/underplums/Documents/work/organic-return-dac/.env


In [ ]:
event_timestamp = datetime(2026, 1, 1)

base_month = event_timestamp.date().replace(day=1).isoformat()
target_month = (pd.Timestamp(base_month) + pd.DateOffset(months=1)).date().isoformat()
feature_date = target_month

print('target:', target)
print('features:', len(features))

In [ ]:
# MLflow remote setup. Если URI уже задан окружением, эта ячейка просто зафиксирует его явно.
MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI')
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print('MLflow tracking URI:', mlflow.get_tracking_uri())
print('MLFLOW_TRACKING_USERNAME exists:', bool(os.getenv('MLFLOW_TRACKING_USERNAME')))
print('MLFLOW_TRACKING_PASSWORD exists:', bool(os.getenv('MLFLOW_TRACKING_PASSWORD')))

In [ ]:
# Optional project state/connections. Для локального parquet train они не обязательны,
# но полезны для MLflow/S3/GP-паттерна проекта.
try:
    try:
        spark.stop()
    except NameError:
        pass

    state = State.from_env()
    engine = state.credentials.loyalty_gp.sa_engine
    s3 = su.get_s3_client()
    print('State loaded')
except Exception as e:
    print('State was not loaded. If train/test parquet is local, this is ok.')
    print(e)


### 2. Load train/test dataset

In [ ]:
train_df = pd.read_parquet(f'train_{base_month}.parquet')
test_df = pd.read_parquet(f'test_{base_month}.parquet')

df = train_df.copy().reset_index(drop=True)
holdout_df = test_df.copy().reset_index(drop=True)

print('df:', train_df.shape)
print('holdout_df:', test_df.shape)

display(df.head())

### 3. Target + DAC segment check

In [ ]:
df['segment'] = df['dac_segment_12m'].astype(str)
holdout_df['segment'] = holdout_df['dac_segment_12m'].astype(str)

df['strat'] = df['segment'].astype(str) + '_' + df[target].astype(str)
holdout_df['strat'] = holdout_df['segment'].astype(str) + '_' + holdout_df[target].astype(str)

strat_counts = df['strat'].value_counts()
if (strat_counts < 5).any():
    print('Only target for CV stratification')
    df['strat'] = df[target].astype(str)

assert target in df.columns, f'Нет target-колонки: {target}'
assert 'contact_id' in df.columns, 'Нет contact_id'
assert df[target].isna().sum() == 0, 'Есть NA в target'
assert df['contact_id'].nunique() == len(df), 'Есть дубли contact_id в train df'
assert holdout_df['contact_id'].nunique() == len(holdout_df), 'Есть дубли contact_id в holdout df'
assert len(set(df['contact_id']) & set(holdout_df['contact_id'])) == 0, 'Есть пересечение contact_id между train и holdout'

print('Train target distribution:')
display(df[target].value_counts(normalize=True).to_frame('share'))
print('Holdout target distribution:')
display(holdout_df[target].value_counts(normalize=True).to_frame('share'))
print('DAC segments:')
display(df.groupby('segment')[target].agg(['count', 'mean']).sort_values('mean', ascending=False))


### 4. Fill recency / basic preprocessing

In [ ]:
recency_cols = [
    'cheque_recency',
    'login_recency',
    'omni_qr_recency',
    'omni_features_recency',
    'perf_recency',
]
for col in recency_cols:
    if col in df.columns:
        df[col] = df[col].fillna(999)
    if holdout_df is not None and col in holdout_df.columns:
        holdout_df[col] = holdout_df[col].fillna(999)

### 5. First baseline: CatBoost on all features

In [ ]:
best_params = {

}

model = CatBoostClassifier(
    random_state=RANDOM_STATE,
    use_best_model=True,
    metric_period=20,
    eval_metric='PRAUC',
    eval_fraction=0.1,
    early_stopping_rounds=50,
    thread_count=-1
    # auto_class_weights='Balanced',
    # **best_params,
)
model

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

df[['class_0', score]] = cross_val_predict(
    model,
    df[features],
    df[target],
    cv=skf.split(df[features], df['strat']),
    method='predict_proba',
)

print(df[[target, score]].head())
print('OOF PR-AUC:', sk_average_precision_score(df[target], df[score]))

In [ ]:
metrics_cv = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X = df[features]
y = df[target]
z = df['strat']

predicts = np.zeros(len(df))

for i, (train_idx, val_idx) in enumerate(skf.split(X, z), start=1):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    print()
    print(f'Training fold {i}...')
    fold_model = CatBoostClassifier(**model.get_params())
    fold_model.fit(X_train, y_train, plot=False, verbose=0)

    print(f'Scoring fold {i}...')
    predicts[val_idx] = fold_model.predict_proba(X_val)[:, 1]

    ap_gain = sk_average_precision_score(y_val, predicts[val_idx]) - y_val.mean()
    metrics_cv.append(ap_gain)
    print(f'Fold {i} complete. AP gain = {ap_gain:.6f}')

print('AP gain by folds:', metrics_cv)
print('AP gain mean:', np.mean(metrics_cv), 'std:', np.std(metrics_cv))

df[score] = predicts

In [ ]:
model.fit(df[features], df[target], plot=False, verbose=1)

df[f'{score}_full_model'] = model.predict_proba(df[features])[:, 1]

if holdout_df is not None:
    holdout_df[score] = model.predict_proba(holdout_df[features])[:, 1]
    print('Holdout scored:', holdout_df.shape)
    display(holdout_df[[target, score]].head())

all_features_model = model
all_features = list(features)


### 6. Evaluate all_features on holdout

In [ ]:
platt = LogisticRegression(C=1e10)
platt_spline = make_pipeline(SplineTransformer(n_knots=5, degree=3), LogisticRegression(C=1.0))
isotonic = IsotonicRegression(out_of_bounds='clip')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X_score = pd.DataFrame(df[score])
y = df[target]
z = df['strat']

X_scored = pd.DataFrame(index=df.index)

for i, (train_idx, val_idx) in enumerate(skf.split(X_score, z), start=1):
    X_train, y_train = X_score.iloc[train_idx], y.iloc[train_idx]
    X_val = X_score.iloc[val_idx]

    print(f'Calibrating on fold {i}...')
    platt.fit(X_train, y_train)
    platt_spline.fit(X_train, y_train)
    isotonic.fit(X_train, y_train)

    X_scored.loc[X_val.index, f'{score}_platt'] = platt.predict_proba(X_val)[:, 1]
    X_scored.loc[X_val.index, f'{score}_platt_spline'] = platt_spline.predict_proba(X_val)[:, 1]
    X_scored.loc[X_val.index, f'{score}_isotonic'] = isotonic.transform(X_val[score])

df[[f'{score}_platt', f'{score}_platt_spline', f'{score}_isotonic']] = X_scored[[f'{score}_platt', f'{score}_platt_spline', f'{score}_isotonic']]
df[[score, f'{score}_platt', f'{score}_platt_spline', f'{score}_isotonic']].describe()

In [ ]:
plt.rcParams['figure.figsize'] = [8, 6]
func.plot_precision_recall_curve(df[target], df[score])

In [ ]:
func.plot_roc(df[target], df[score])

In [ ]:
plt.figure(figsize=(12, 8))
plt.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration', alpha=0.5)

models_to_plot = [
    (df[score], 'Raw', 'blue'),
    (df[f'{score}_platt'], 'Platt', 'green'),
    (df[f'{score}_platt_spline'], 'Platt Spline', 'purple'),
    (df[f'{score}_isotonic'], 'Isotonic', 'orange'),
]

for preds, label, color in models_to_plot:
    prob_true, prob_pred = calibration_curve(df[target], preds, n_bins=10, strategy='quantile')
    plt.plot(prob_pred, prob_true, marker='o', label=label, color=color)

plt.xlabel('Mean predicted probability')
plt.ylabel('Observed target rate')
plt.title('Calibration Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df[df[target] == 0][score], label='No churn from DAC', kde=True, stat='density', color='blue')
sns.histplot(df[df[target] == 1][score], label='Churn from DAC', kde=True, stat='density', alpha=0.25, color='orange')
plt.title('Labeled Score Histogram')
plt.legend();

In [ ]:
if holdout_df is not None:
    print('Holdout PR-AUC:', sk_average_precision_score(holdout_df[target], holdout_df[score]))
    print('Holdout ROC-AUC:', sk_roc_auc_score(holdout_df[target], holdout_df[score]))
    func.plot_precision_recall_curve(holdout_df[target], holdout_df[score], title_suffix='Holdout')

In [ ]:
def find_threshold(y_true, y_pred, eval_function, **kwargs):
    score_by_threshold = {}
    for t in np.arange(0.05, 1, 0.05):
        y_class = y_pred > t
        score_by_threshold[t] = eval_function(y_true, y_class, **kwargs)
    return max(score_by_threshold.items(), key=lambda x: x[1])[0]


t = df[target]
s = df[score]

for beta in np.arange(0.05, 1, 0.05):
    best_t = find_threshold(t, s, fbeta_score, beta=beta)
    c_beta = s > best_t
    print(round(beta, 2), 'threshold:', round(best_t, 3), 'predicted_positive_share:', round(c_beta.mean(), 4))

In [ ]:
threshold_f1 = find_threshold(t, s, fbeta_score, beta=1)
threshold_f05 = find_threshold(t, s, fbeta_score, beta=0.5)
threshold_f035 = find_threshold(t, s, fbeta_score, beta=0.35)

print('threshold from parameters.py:', threshold)
print('threshold F1:', threshold_f1)
print('threshold F0.5:', threshold_f05)
print('threshold F0.35:', threshold_f035)

selected_threshold = threshold_f1
c = s > selected_threshold
print(classification_report(t, c))

In [ ]:
metrics = {
    'ROC-AUC': roc_auc_score(t, s),
    'PR-AUC': precision_recall_auc(t, s),
    'AP gain': sk_average_precision_score(t, s) - t.mean(),
    'Log loss': log_loss(t, s),
    'Recall': recall_score(t, c),
    'Precision': precision_score(t, c),
    'Specificity': sensitivity_specificity(t, c)[1],
    'Balanced accuracy': balanced_accuracy(t, c),
    'F1': f1_score(t, c),
    'F0.5': fbeta_score(t, c, beta=0.5),
    'F2': fbeta_score(t, c, beta=2),
    'Brier score': brier_score_loss(t, s),
    'Youden j': youden_j(t, c),
    'Markedness': markedness(t, s),
    'Lift': lift(t, s),
    'Gini': gini(t, s),
    'Kolmogorov-Smirnov statistic': ks_stat_bin_class(t, s),
    'Expected Calibration Error': ece_mce_fast(t, s)[0],
    'Maximum Calibration Error': ece_mce_fast(t, s)[1],
    'Matthews corrcoef': matthews_corrcoef(t, c),
    'Cohen kappa score': cohen_kappa_score(t, c),
}

pd.DataFrame(metrics, index=['value']).T

In [ ]:
cm = confusion_matrix(t, c)
class_labels = ['No churn from DAC', 'Churn from DAC']
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.title('Confusion Matrix');

In [ ]:
print('Metrics by DAC segment:')
rows = []
for segment, part in df.groupby('segment'):
    if part[target].nunique() < 2:
        continue
    rows.append({
        'segment': segment,
        'rows': len(part),
        'target_rate': part[target].mean(),
        'roc_auc': sk_roc_auc_score(part[target], part[score]),
        'pr_auc': sk_average_precision_score(part[target], part[score]),
        'ap_gain': sk_average_precision_score(part[target], part[score]) - part[target].mean(),
    })

pd.DataFrame(rows).sort_values('pr_auc', ascending=False)

### 7. Feature importance / permutation importance

In [ ]:
X_train, X_val, y_train, y_val, z_train, _ = train_test_split(
    df[features],
    df[target],
    df['strat'],
    stratify=df['strat'],
    test_size=0.3,
    random_state=RANDOM_STATE,
)

model.fit(X_train, y_train, plot=False, verbose=1)
predicts_val = model.predict_proba(X_val)[:, 1]

print('Val PR-AUC:', sk_average_precision_score(y_val, predicts_val))
print('Val ROC-AUC:', sk_roc_auc_score(y_val, predicts_val))

In [ ]:
shap_val_sample_size = min(200_000, len(X_val))
X_val_sample = X_val.sample(shap_val_sample_size, random_state=RANDOM_STATE)
explainer = shap.TreeExplainer(model)
shap_vals_val = explainer.shap_values(X_val_sample)
shap.summary_plot(shap_vals_val, X_val_sample)

In [ ]:
RUN_PERMUTATION_IMPORTANCE = True

if RUN_PERMUTATION_IMPORTANCE:
    r = permutation_importance(
        model,
        X_val,
        y_val,
        scoring='average_precision',
        n_repeats=10,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    pi_df = pd.DataFrame({
        'feature': X_val.columns,
        'importance_mean': r.importances_mean,
        'importance_std': r.importances_std,
    }).sort_values('importance_mean', ascending=False)
    display(pi_df.head(100))

    pi_filter = pi_df.loc[pi_df['importance_mean'] < 0.0001, 'feature'].tolist()
    joblib.dump(pi_filter, 'pi_filter_churn_from_dac.pkl')

In [ ]:
interactions = model.get_feature_importance(type='Interaction')
interaction_df = pd.DataFrame(interactions, columns=['f1', 'f2', 'score'])
interaction_df['f1_name'] = interaction_df['f1'].apply(lambda x: features[int(x)])
interaction_df['f2_name'] = interaction_df['f2'].apply(lambda x: features[int(x)])
top_interactions = interaction_df.sort_values(by='score', ascending=False).head(15)
display(top_interactions[['f1_name', 'f2_name', 'score']])

### 8. Redundant features analysis

In [ ]:
reduntant_filter, pairs = func.get_redundant_features(X_train, threshold=0.85)
print('Redundant features:', len(reduntant_filter))
display(pd.DataFrame(pairs, columns=['feature_1', 'feature_2', 'corr']).head(100))

joblib.dump(reduntant_filter, 'reduntant_filter_churn_from_dac.pkl')

### 9. Feature set experiments

#### 9.1 all_features
#### 9.2 no_redundant_features
#### 9.3 top_80_features
#### 9.4 no_dac_segment_features
#### 9.5 only_dac_segment_features

In [ ]:
def get_top_features_by_importance(base_features, top_n=80):
    base_features = list(dict.fromkeys(base_features))

    if 'pi_df' in globals() and isinstance(pi_df, pd.DataFrame) and {'feature', 'importance_mean'} <= set(pi_df.columns):
        top_features = (
            pi_df[pi_df['feature'].isin(base_features)]
            .sort_values('importance_mean', ascending=False)['feature']
            .head(top_n)
            .tolist()
        )
        source = 'permutation_importance'
    else:
        try:
            fi_df = pd.DataFrame({
                'feature': base_features,
                'importance': model.get_feature_importance(),
            }).sort_values('importance', ascending=False)
            top_features = fi_df['feature'].head(top_n).tolist()
            source = 'catboost_feature_importance'
        except Exception:
            top_features = base_features[:top_n]
            source = 'original_order_fallback'

    return top_features, source


def get_dac_segment_features(base_features):
    # DAC-history / segment фичи. Это не leakage, если они посчитаны только до target_month.
    dac_exact = {
        'dac_age_months',
        'dac_months_count',
        'dac_months_per_dac_age_ratio',
        'dac_months_last_3',
        'dac_months_last_6',
        'dac_months_last_12',
        'dac_share_last_12',
        'is_stable_dac',
        'is_regular_dac',
        'is_unstable_dac',
        'is_new_dac',
        'segment',
        'dac_segment_12m',
    }
    return [f for f in base_features if f in dac_exact or f.startswith('dac_') or f.endswith('_dac') or f.startswith('is_') and f.endswith('_dac')]


base_features = [f for f in list(dict.fromkeys(features)) if f in df.columns and f in holdout_df.columns]
top_80_features, top_80_source = get_top_features_by_importance(base_features, top_n=80)
redundant_features = [f for f in globals().get('reduntant_filter', []) if f in base_features]
dac_segment_features = get_dac_segment_features(base_features)

feature_sets = {
    '01_all_features': base_features,
    '02_no_redundant_features': [f for f in base_features if f not in redundant_features],
    '03_top_80_features': top_80_features,
    '04_no_dac_segment_features': [f for f in base_features if f not in dac_segment_features],
    '05_only_dac_segment_features': dac_segment_features,
}

feature_sets = {name: list(dict.fromkeys(cols)) for name, cols in feature_sets.items()}

feature_set_summary = pd.DataFrame([
    {'feature_set': name, 'features_count': len(cols), 'features': ', '.join(cols[:20]) + (' ...' if len(cols) > 20 else '')}
    for name, cols in feature_sets.items()
])

print('Top-80 source:', top_80_source)
print('Redundant features:', len(redundant_features))
print('DAC segment/history features:', len(dac_segment_features), dac_segment_features)
display(feature_set_summary)

assert len(feature_sets['05_only_dac_segment_features']) > 0, 'Не нашлись DAC segment/history фичи'

In [ ]:
def fbeta_threshold(y_true, y_score, beta=1.0):
    scores = {}
    for thr in np.arange(0.05, 1.0, 0.05):
        scores[thr] = fbeta_score(y_true, y_score > thr, beta=beta)
    return max(scores.items(), key=lambda x: x[1])[0]


def decile_report(data, score_col, target_col):
    tmp = data[[score_col, target_col]].copy()
    tmp['score_decile'] = pd.qcut(
        tmp[score_col].rank(method='first'),
        q=10,
        labels=False,
        duplicates='drop',
    ) + 1
    return tmp.groupby('score_decile').agg(
        rows=(target_col, 'size'),
        target_rate=(target_col, 'mean'),
        score_min=(score_col, 'min'),
        score_max=(score_col, 'max'),
    ).sort_index(ascending=False)


def calculate_binary_metrics(y_true, y_score, threshold_value):
    y_pred = y_score > threshold_value
    return {
        'roc_auc': float(sk_roc_auc_score(y_true, y_score)),
        'pr_auc': float(sk_average_precision_score(y_true, y_score)),
        'ap_gain': float(sk_average_precision_score(y_true, y_score) - np.mean(y_true)),
        'threshold': float(threshold_value),
        'predicted_positive_share': float(np.mean(y_pred)),
        'precision': float(precision_score(y_true, y_pred)),
        'recall': float(recall_score(y_true, y_pred)),
        'f1': float(f1_score(y_true, y_pred)),
        'f05': float(fbeta_score(y_true, y_pred, beta=0.5)),
        'f2': float(fbeta_score(y_true, y_pred, beta=2)),
        'brier': float(sk_brier_score_loss(y_true, y_score)),
    }


def train_score_feature_set(feature_set_name, feature_cols, model_params=None, log_to_mlflow=True):
    feature_cols = list(dict.fromkeys([f for f in feature_cols if f in df.columns and f in holdout_df.columns]))
    assert len(feature_cols) > 0, f'Пустой список фичей для {feature_set_name}'

    params_model = {
        'random_state': RANDOM_STATE,
        'use_best_model': True,
        'metric_period': 20,
        'eval_metric': 'PRAUC',
        'eval_fraction': 0.1,
        'early_stopping_rounds': 50,
        'thread_count': -1,
        'verbose': -1,
    }
    if model_params:
        params_model.update(model_params)

    exp_model = CatBoostClassifier(**params_model)
    exp_model.fit(df[feature_cols], df[target], plot=False, verbose=0)

    train_score_col = f'{score}_{feature_set_name}_train'
    holdout_score_col = f'{score}_{feature_set_name}'

    df[train_score_col] = exp_model.predict_proba(df[feature_cols])[:, 1]
    holdout_df[holdout_score_col] = exp_model.predict_proba(holdout_df[feature_cols])[:, 1]

    selected_thr = fbeta_threshold(df[target], df[train_score_col], beta=1)
    train_metrics = calculate_binary_metrics(df[target], df[train_score_col], selected_thr)
    holdout_metrics = calculate_binary_metrics(holdout_df[target], holdout_df[holdout_score_col], selected_thr)

    deciles = decile_report(holdout_df, holdout_score_col, target)
    top_decile_target_rate = float(deciles.iloc[0]['target_rate'])
    top_decile_lift = float(top_decile_target_rate / holdout_df[target].mean())

    segment_rows = []
    for segment_name, part in holdout_df.groupby('segment'):
        if part[target].nunique() < 2:
            continue
        segment_rows.append({
            'segment': segment_name,
            'rows': len(part),
            'target_rate': float(part[target].mean()),
            'pr_auc': float(sk_average_precision_score(part[target], part[holdout_score_col])),
            'roc_auc': float(sk_roc_auc_score(part[target], part[holdout_score_col])),
        })
    segment_metrics = pd.DataFrame(segment_rows)

    row = {
        'feature_set': feature_set_name,
        'features_count': len(feature_cols),
        'train_pr_auc': train_metrics['pr_auc'],
        'holdout_pr_auc': holdout_metrics['pr_auc'],
        'holdout_roc_auc': holdout_metrics['roc_auc'],
        'holdout_ap_gain': holdout_metrics['ap_gain'],
        'holdout_f1': holdout_metrics['f1'],
        'holdout_precision': holdout_metrics['precision'],
        'holdout_recall': holdout_metrics['recall'],
        'threshold': selected_thr,
        'top_decile_target_rate': top_decile_target_rate,
        'top_decile_lift': top_decile_lift,
    }

    if log_to_mlflow:
        experiment_name = f'{project_name}_{model_type}_{jira}_feature_selection'
        mlflow.set_experiment(experiment_name)
        run_name = f'{feature_set_name}_{base_month}_features_{len(feature_cols)}'

        features_file = f'features_{feature_set_name}.txt'
        deciles_file = f'deciles_{feature_set_name}.csv'
        segments_file = f'segment_metrics_{feature_set_name}.csv'
        model_file = f'model_{feature_set_name}.cbm'

        Path(features_file).write_text('\n'.join(feature_cols), encoding='utf-8')
        deciles.to_csv(deciles_file)
        segment_metrics.to_csv(segments_file, index=False)
        exp_model.save_model(model_file)

        with mlflow.start_run(run_name=run_name, description=f'Feature set experiment: {feature_set_name}'):
            mlflow.log_param('base_month', base_month)
            mlflow.log_param('target_month', target_month)
            mlflow.log_param('feature_set', feature_set_name)
            mlflow.log_param('features_count', len(feature_cols))
            mlflow.log_param('top_80_source', top_80_source)
            mlflow.log_param('removed_redundant_features_count', len(redundant_features) if feature_set_name == '03_no_redundant_features' else 0)
            mlflow.log_param('dac_segment_features_count', len(dac_segment_features))
            mlflow.log_param('threshold', selected_thr)

            for k, v in params_model.items():
                if isinstance(v, (str, int, float, bool)) or v is None:
                    mlflow.log_param(f'catboost_{k}', v)

            for k, v in row.items():
                if k not in {'feature_set'}:
                    mlflow.log_metric(k, float(v))

            for _, seg_row in segment_metrics.iterrows():
                seg = str(seg_row['segment']).replace('-', '_').replace(' ', '_')
                mlflow.log_metric(f'segment_{seg}_pr_auc', float(seg_row['pr_auc']))
                mlflow.log_metric(f'segment_{seg}_roc_auc', float(seg_row['roc_auc']))

            mlflow.log_artifact(features_file)
            mlflow.log_artifact(deciles_file)
            mlflow.log_artifact(segments_file)
            mlflow.log_artifact(model_file)

        for local_file in [features_file, deciles_file, segments_file, model_file]:
            try:
                Path(local_file).unlink()
            except FileNotFoundError:
                pass

    return row, exp_model, feature_cols, deciles, segment_metrics

In [ ]:
RUN_FEATURE_SET_EXPERIMENTS = True

feature_experiment_rows = []
feature_experiment_models = {}
feature_experiment_features = {}
feature_experiment_deciles = {}
feature_experiment_segments = {}

if RUN_FEATURE_SET_EXPERIMENTS:
    for feature_set_name, feature_cols in feature_sets.items():
        print(f'\n=== {feature_set_name}: {len(feature_cols)} features ===')
        row, exp_model, used_features, deciles, segment_metrics = train_score_feature_set(
            feature_set_name,
            feature_cols,
            model_params=None,
            log_to_mlflow=True,
        )
        feature_experiment_rows.append(row)
        feature_experiment_models[feature_set_name] = exp_model
        feature_experiment_features[feature_set_name] = used_features
        feature_experiment_deciles[feature_set_name] = deciles
        feature_experiment_segments[feature_set_name] = segment_metrics
        display(pd.DataFrame([row]))


### 10. Compare experiments

In [ ]:
feature_experiment_results = (
    pd.DataFrame(feature_experiment_rows)
    .sort_values('holdout_pr_auc', ascending=False)
    .reset_index(drop=True)
)

display(feature_experiment_results)

plt.figure(figsize=(10, 5))
sns.barplot(
    data=feature_experiment_results,
    x='holdout_pr_auc',
    y='feature_set',
    color='steelblue',
)
plt.title('Feature Set Comparison: Holdout PR-AUC')
plt.xlabel('Holdout PR-AUC')
plt.ylabel('Feature set')
plt.grid(axis='x', alpha=0.3)
plt.show()


### 11. Select best feature set

In [ ]:
best_feature_set_name = feature_experiment_results.iloc[0]['feature_set']
best_feature_set_features = feature_experiment_features[best_feature_set_name]
best_feature_set_model = feature_experiment_models[best_feature_set_name]

print('Best feature set:', best_feature_set_name)
print('Best features count:', len(best_feature_set_features))
print('Best holdout PR-AUC:', feature_experiment_results.iloc[0]['holdout_pr_auc'])

display(pd.DataFrame({'feature': best_feature_set_features}))


### 12. Optuna tuning only for best feature set

In [ ]:
# Если optuna не установлена в kernel, один раз выполни:
# %pip install optuna

import optuna

In [ ]:
RUN_OPTUNA_TUNING = True
N_TRIALS = 30
OPTUNA_TIMEOUT = None  # например, 60 * 60 для ограничения в 1 час

X_train_tune, X_val_tune, y_train_tune, y_val_tune, z_train_tune, _ = train_test_split(
    df[best_feature_set_features],
    df[target],
    df['strat'],
    stratify=df['strat'],
    test_size=0.25,
    random_state=RANDOM_STATE,
)


def objective(trial):
    params_trial = {
        'random_state': RANDOM_STATE,
        'loss_function': 'Logloss',
        'eval_metric': 'PRAUC',
        'iterations': trial.suggest_int('iterations', 300, 1200),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 30.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 0.1, 10.0, log=True),
        'border_count': trial.suggest_categorical('border_count', [64, 128, 254]),
        'early_stopping_rounds': 50,
        'thread_count': -1,
        'verbose': False,
    }

    bootstrap_type = trial.suggest_categorical('bootstrap_type', ['Bayesian', 'Bernoulli', 'MVS'])
    params_trial['bootstrap_type'] = bootstrap_type

    if bootstrap_type == 'Bayesian':
        params_trial['bagging_temperature'] = trial.suggest_float('bagging_temperature', 0.0, 5.0)
    else:
        params_trial['subsample'] = trial.suggest_float('subsample', 0.6, 1.0)

    tune_model = CatBoostClassifier(**params_trial)
    tune_model.fit(
        X_train_tune,
        y_train_tune,
        eval_set=(X_val_tune, y_val_tune),
        use_best_model=True,
        verbose=False,
    )

    val_pred = tune_model.predict_proba(X_val_tune)[:, 1]
    return sk_average_precision_score(y_val_tune, val_pred)


if RUN_OPTUNA_TUNING:
    study = optuna.create_study(
        direction='maximize',
        study_name=f'catboost_{best_feature_set_name}_churn_from_dac_pr_auc',
    )
    study.optimize(objective, n_trials=N_TRIALS, timeout=OPTUNA_TIMEOUT, show_progress_bar=True)

    print('Best Optuna validation PR-AUC:', study.best_value)
    print('Best params:')
    display(study.best_params)
else:
    study = None

In [ ]:
if RUN_OPTUNA_TUNING:
    best_catboost_params = {
        'random_state': RANDOM_STATE,
        'loss_function': 'Logloss',
        'eval_metric': 'PRAUC',
        'early_stopping_rounds': 50,
        'thread_count': -1,
        'verbose': False,
        **study.best_params,
    }

    tuned_feature_set_name = f'06_optuna_{best_feature_set_name}'
    tuned_row, tuned_model, tuned_features, tuned_deciles, tuned_segment_metrics = train_score_feature_set(
        tuned_feature_set_name,
        best_feature_set_features,
        model_params=best_catboost_params,
        log_to_mlflow=True,
    )

    feature_experiment_results = pd.concat(
        [feature_experiment_results, pd.DataFrame([tuned_row])],
        ignore_index=True,
    ).sort_values('holdout_pr_auc', ascending=False)

    feature_experiment_models[tuned_feature_set_name] = tuned_model
    feature_experiment_features[tuned_feature_set_name] = tuned_features
    feature_experiment_deciles[tuned_feature_set_name] = tuned_deciles
    feature_experiment_segments[tuned_feature_set_name] = tuned_segment_metrics

    print('Final feature experiment results:')
    display(feature_experiment_results)

    final_best_feature_set_name = feature_experiment_results.iloc[0]['feature_set']
    final_best_features = feature_experiment_features[final_best_feature_set_name]
    final_best_model = feature_experiment_models[final_best_feature_set_name]

    print('Final best feature set:', final_best_feature_set_name)
    print('Final best features count:', len(final_best_features))
    print('Final best holdout PR-AUC:', feature_experiment_results.iloc[0]['holdout_pr_auc'])

    joblib.dump(study.best_params, 'best_catboost_params_churn_from_dac.pkl')
    final_best_model.save_model('best_feature_set_model_churn_from_dac.cbm')
    Path('best_feature_set_features.txt').write_text('\n'.join(final_best_features), encoding='utf-8')
else:
    final_best_feature_set_name = best_feature_set_name
    final_best_features = best_feature_set_features
    final_best_model = best_feature_set_model

### 13. Final model evaluation

In [ ]:
final_score_col = 'final_best_score'
holdout_df[final_score_col] = final_best_model.predict_proba(holdout_df[final_best_features])[:, 1]
df[final_score_col] = final_best_model.predict_proba(df[final_best_features])[:, 1]

final_threshold = fbeta_threshold(df[target], df[final_score_col], beta=1)
final_holdout_metrics = calculate_binary_metrics(holdout_df[target], holdout_df[final_score_col], final_threshold)
final_train_metrics = calculate_binary_metrics(df[target], df[final_score_col], final_threshold)

final_deciles = decile_report(holdout_df, final_score_col, target)
final_segment_rows = []
for segment_name, part in holdout_df.groupby('segment'):
    if part[target].nunique() < 2:
        continue
    final_segment_rows.append({
        'segment': segment_name,
        'rows': len(part),
        'target_rate': float(part[target].mean()),
        'pr_auc': float(sk_average_precision_score(part[target], part[final_score_col])),
        'roc_auc': float(sk_roc_auc_score(part[target], part[final_score_col])),
    })
final_segment_metrics = pd.DataFrame(final_segment_rows).sort_values('pr_auc', ascending=False)

print('Final best feature set:', final_best_feature_set_name)
print('Final features count:', len(final_best_features))
print('Final threshold:', final_threshold)
print('Final holdout metrics:')
display(pd.DataFrame(final_holdout_metrics, index=['holdout']).T)

print('Final decile report:')
display(final_deciles)

print('Final segment metrics:')
display(final_segment_metrics)

func.plot_precision_recall_curve(holdout_df[target], holdout_df[final_score_col], title_suffix='Final Holdout')
func.plot_roc(holdout_df[target], holdout_df[final_score_col], title_suffix='Final Holdout')


### 14. Log final model to MLflow

In [ ]:
final_model_name = f'{project_name}_{model_type}'
final_experiment_name = f'{project_name}_{model_type}_{jira}_final'
mlflow.set_experiment(final_experiment_name)

final_run_name = f'final_{final_best_feature_set_name}_{base_month}'
final_signature = infer_signature(holdout_df.head(1)[final_best_features], holdout_df.head(1)[final_score_col])

Path('final_best_features.txt').write_text('\\n'.join(final_best_features), encoding='utf-8')
final_segment_metrics.to_csv('final_segment_metrics.csv', index=False)
final_deciles.to_csv('final_deciles.csv')
feature_experiment_results.to_csv('feature_experiment_results.csv', index=False)

final_best_model.save_model('final_best_model_churn_from_dac.cbm')

with mlflow.start_run(run_name=final_run_name, description='Final selected churn-from-DAC model'):
    mlflow.log_param('base_month', base_month)
    mlflow.log_param('target_month', target_month)
    mlflow.log_param('final_feature_set', final_best_feature_set_name)
    mlflow.log_param('features_count', len(final_best_features))
    mlflow.log_param('threshold', final_threshold)
    mlflow.log_param('target_rate_train', float(df[target].mean()))
    mlflow.log_param('target_rate_holdout', float(holdout_df[target].mean()))

    if 'best_catboost_params' in globals():
        for k, v in best_catboost_params.items():
            if isinstance(v, (str, int, float, bool)) or v is None:
                mlflow.log_param(f'catboost_{k}', v)

    for k, v in final_train_metrics.items():
        mlflow.log_metric(f'train_{k}', float(v))
    for k, v in final_holdout_metrics.items():
        mlflow.log_metric(f'holdout_{k}', float(v))

    for _, seg_row in final_segment_metrics.iterrows():
        seg = str(seg_row['segment']).replace('-', '_').replace(' ', '_')
        mlflow.log_metric(f'holdout_segment_{seg}_pr_auc', float(seg_row['pr_auc']))
        mlflow.log_metric(f'holdout_segment_{seg}_roc_auc', float(seg_row['roc_auc']))

    for path in [
        'final_best_features.txt',
        'final_segment_metrics.csv',
        'final_deciles.csv',
        'feature_experiment_results.csv',
        'final_best_model_churn_from_dac.cbm',
    ]:
        mlflow.log_artifact(path)

    mlflow.sklearn.log_model(
        final_best_model,
        name=final_model_name,
        signature=final_signature,
        registered_model_name=final_model_name,
    )

print('Logged final model to:', final_experiment_name)
print('Registered model:', final_model_name)

### 15. Save artifacts

In [ ]:
Path(artifacts_dir).mkdir(parents=True, exist_ok=True)

# Clean artifacts dir
for file in Path(artifacts_dir).iterdir():
    if file.is_file():
        try:
            file.unlink()
        except Exception as e:
            print(f'Failed to delete {file}. Reason: {e}')

# Final model artifacts
Path(os.path.join(artifacts_dir, 'final_best_features.txt')).write_text('\\n'.join(final_best_features), encoding='utf-8')
feature_experiment_results.to_csv(os.path.join(artifacts_dir, 'feature_experiment_results.csv'), index=False)
final_segment_metrics.to_csv(os.path.join(artifacts_dir, 'final_segment_metrics.csv'), index=False)
final_deciles.to_csv(os.path.join(artifacts_dir, 'final_deciles.csv'))
final_best_model.save_model(os.path.join(artifacts_dir, 'final_best_model_churn_from_dac.cbm'))

# Save scored datasets
scored_path = f'df_churn_from_dac_scored_{base_month}.parquet'
holdout_scored_path = f'holdout_churn_from_dac_scored_{base_month}.parquet'
df.to_parquet(scored_path, index=False)
holdout_df.to_parquet(holdout_scored_path, index=False)

print('Artifacts dir:', Path(artifacts_dir).resolve())
print('Saved:', scored_path)
print('Saved:', holdout_scored_path)

### 16. Experiment report

In [ ]:
experiment_report = {
    'event_timestamp': event_timestamp,
    'base_month': base_month,
    'target_month': target_month,
    'target': target,
    'train_rows': len(df),
    'holdout_rows': len(holdout_df),
    'target_rate_train': df[target].mean(),
    'target_rate_holdout': holdout_df[target].mean(),
    'final_feature_set': final_best_feature_set_name,
    'final_features_count': len(final_best_features),
    'final_threshold': final_threshold,
    'final_holdout_pr_auc': final_holdout_metrics['pr_auc'],
    'final_holdout_roc_auc': final_holdout_metrics['roc_auc'],
    'final_holdout_f1': final_holdout_metrics['f1'],
}

print('Experiment report:')
display(pd.DataFrame(experiment_report, index=['value']).T)

print('Top final feature importances:')
try:
    final_fi = pd.DataFrame({
        'feature': final_best_features,
        'importance': final_best_model.get_feature_importance(),
    }).sort_values('importance', ascending=False)
    display(final_fi.head(40))
except Exception as e:
    print('Feature importance is not available yet:', e)
